# Train a RoBERTa Classifier on the GPT 5.4 mini Data

Note: this code was tested in Google Colab using an A100 GPU, and has not been verified in any other scenarios

In [ ]:
import os

try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
  gpt_5_4_mini_roberta_dir = "/content/drive/MyDrive/Colab Notebooks/NLU_project/gpt-5-4-mini-roberta-classifier" # Directory to store intermediate and final versions of the RoBERTa model trained on GPT 5.4 mini data
except:
  pass

# Set up directory to store intermediate and final versions of the RoBERTa model trained on GPT 5.4 mini data
# Use Google Drive if available, otherwise save locally
if os.path.exists('/content/drive/MyDrive/Colab Notebooks/NLU_project/'):
  gpt5_4_mini_roberta_dir = "/content/drive/MyDrive/Colab Notebooks/NLU_project/gpt-5-4-mini-roberta-classifier"
else:
  gpt_5_4_mini_roberta_dir = "./gpt-5-4-mini-roberta-classifier"

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

# Download the data from GitHub
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/Processed_GPT5.4-mini_Data/train.csv
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/Processed_GPT5.4-mini_Data/test.csv
!wget https://raw.githubusercontent.com/Lennart-Martens/Explaining-Uncanny-Text/refs/heads/main/Processed_GPT5.4-mini_Data/val.csv

# load in the GPT 5.4 mini data
gpt_5_4_mini_dataset = load_dataset("csv", data_files={
    "train":      "train.csv",
    "test":       "test.csv",
    "val":        "val.csv",
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [4]:
from transformers import RobertaTokenizer

gpt_5_4_mini_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(batch):
    return gpt_5_4_mini_tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

gpt_5_4_mini_tokenized_data = gpt_5_4_mini_dataset.map(tokenize, batched=True)
gpt_5_4_mini_tokenized_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/85973 [00:00<?, ? examples/s]

Map:   0%|          | 0/11080 [00:00<?, ? examples/s]

Map:   0%|          | 0/11044 [00:00<?, ? examples/s]

In [5]:
from transformers import RobertaForSequenceClassification

gpt_5_4_mini_roberta = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Set up method to use when computing metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

# Set up training arguments
training_args = TrainingArguments(
    output_dir=gpt_5_4_mini_roberta_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=250,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trnr = Trainer(
    model=gpt_5_4_mini_roberta,
    args=training_args,
    train_dataset=gpt_5_4_mini_tokenized_data["train"],
    eval_dataset=gpt_5_4_mini_tokenized_data["val"],
    compute_metrics=compute_metrics,
)

trnr.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.018537,0.029633,0.993390,0.993391
2,0.010850,0.050144,0.991217,0.991220
3,0.015927,0.040448,0.994748,0.994749


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=16122, training_loss=0.030180184872727795, metrics={'train_runtime': 1154.0036, 'train_samples_per_second': 223.499, 'train_steps_per_second': 13.97, 'total_flos': 6.786134028739584e+16, 'train_loss': 0.030180184872727795, 'epoch': 3.0})

In [7]:
# Evaluate on test set after training
trnr.evaluate(gpt_5_4_mini_tokenized_data["test"])

{'eval_loss': 0.041078388690948486,
 'eval_accuracy': 0.9943140794223827,
 'eval_f1': 0.9943147046920061,
 'eval_runtime': 11.5209,
 'eval_samples_per_second': 961.733,
 'eval_steps_per_second': 30.119,
 'epoch': 3.0}

In [8]:
# Save the model and tokenizer to avoid needing to train again

gpt_5_4_mini_roberta.save_pretrained(gpt_5_4_mini_roberta_dir)
gpt_5_4_mini_tokenizer.save_pretrained(gpt_5_4_mini_roberta_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab Notebooks/NLU_project/gpt-5-4-mini-roberta-classifier/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/NLU_project/gpt-5-4-mini-roberta-classifier/tokenizer.json')